# Exploration des données - Collection Équinoxe

Notebook de travail, séparé du notebook final (`equinoxe-2026.ipynb`). Son rôle :

1. **Vérifier ce que chaque colonne représente réellement** - la description ne correspond pas toujours aux données.
2. **Tester les relations** qui pourraient expliquer la hausse et les concessions.
3. **Repérer les problèmes de qualité** avant de calculer quoi que ce soit.

Les conclusions retenues seront reportées (en version courte) dans le notebook final.

> Rappel : l'extrait contient une part fixe des unités de chaque immeuble. On n'y calcule **jamais**
> d'occupation, d'inoccupation, d'absorption ni de rôle de loyers - seulement des variations de loyer.

## 0. Configuration et chargement

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 50)

LISTINGS_PATH = "equinoxe_listings.csv"
LEASES_PATH = "equinoxe_lease_history.csv"
CONCESSIONS_PATH = "equinoxe_concessions.csv"
ASKING_PATH = "equinoxe_asking_history.csv"

UNIT_KEY = ["sPropCode", "sUnitCode"]   # jamais sSite + sUnitCode
RENEWAL, TURNOVER = 1, 0

FIRST_ANALYSIS_YEAR = 2019
LAST_OBSERVED_YEAR = 2025

# Tolérance (en $ par mois) pour dire que deux loyers « concordent » : on accepte un arrondi au dollar.
RENT_MATCH_TOLERANCE = 1.0

# Codes de concessions qui ne touchent pas le loyer lui-même, mais un accessoire.
ACCESSORY_CHARGE_CODES = ["freepark", "freelock", "freeappl"]

# Tranches de durée entre deux baux d'une même unité (jours), pour le test vacance -> concession.
DOWNTIME_BINS_DAYS = [0, 7, 30, 60, np.inf]

PCT = 100

listings = pd.read_csv(LISTINGS_PATH)
leases = pd.read_csv(LEASES_PATH)
concessions = pd.read_csv(CONCESSIONS_PATH)
asking = pd.read_csv(ASKING_PATH)

### Conversion des types

On convertit toutes les dates une fois. `lease_start_year` sert à rattacher chaque bail à une année.

In [ ]:
for col in ["sLeaseFrom", "sLeaseTo", "sSignDate", "sAvailable"]:
    leases[col] = pd.to_datetime(leases[col])
for col in ["sDateFrom", "sDateTo"]:
    concessions[col] = pd.to_datetime(concessions[col])
asking["month_start"] = pd.to_datetime(asking["sMonth"] + "-01")

leases["lease_start_year"] = leases["sLeaseFrom"].dt.year

## 1. Vue d'ensemble des tables

Taille, période couverte et valeurs manquantes de chaque table.

In [ ]:
tables = {"listings": listings, "leases": leases, "concessions": concessions, "asking": asking}
overview = pd.DataFrame({
    name: {"rows": len(df), "columns": df.shape[1], "missing_cells": int(df.isna().sum().sum())}
    for name, df in tables.items()
}).T
print(overview, end="\n\n")
print("Baux :", leases["sLeaseFrom"].min().date(), "->", leases["sLeaseFrom"].max().date())
print("Loyers affichés :", asking["sMonth"].min(), "->", asking["sMonth"].max())
print("Concessions :", concessions["sDateFrom"].min().date(), "->", concessions["sDateTo"].max().date())

In [ ]:
leases_per_building_year = pd.crosstab(leases["sBuilding"], leases["lease_start_year"])
leases_per_building_year

## 2. Les clés d'unité

On vérifie quelles colonnes identifient vraiment un appartement :

- `sPropCode + sUnitCode` doit correspondre à exactement un `hUnit` et un `sLink`;
- `sSite + sUnitCode` ne doit **pas** être utilisé (collisions annoncées dans les consignes);
- toutes les unités des baux doivent exister dans `listings`.

In [ ]:
hunits_per_key = leases.groupby(UNIT_KEY)["hUnit"].nunique().max()
keys_per_hunit = leases.groupby("hUnit")[UNIT_KEY].nunique().max().max()
keys_per_link = leases.groupby("sLink")[UNIT_KEY].nunique().max().max()
print(f"hUnit par clé sPropCode+sUnitCode (max) : {hunits_per_key}")
print(f"clés par hUnit (max) : {keys_per_hunit}")
print(f"clés par sLink (max) : {keys_per_link}")

site_collisions = listings[listings.duplicated(["sSite", "sUnitCode"], keep=False)]
print(f"unités en collision sur sSite + sUnitCode : {len(site_collisions)} / {len(listings)}")

lease_units = leases[UNIT_KEY].drop_duplicates()
missing_from_listings = lease_units.merge(listings[UNIT_KEY], how="left", indicator=True)
print("unités des baux absentes de listings :", (missing_from_listings["_merge"] == "left_only").sum())
print("baux en double (même unité, même date de début) :", leases.duplicated(UNIT_KEY + ["sLeaseFrom"]).sum())

**Lecture :** `sPropCode + sUnitCode`, `hUnit` et `sLink` sont trois clés équivalentes (relation 1 pour 1).
`sSite + sUnitCode` provoque 248 collisions : à ne jamais utiliser. Aucun bail orphelin, aucun doublon.

## 3. Que représente chaque colonne des baux?

On teste les colonnes dont le sens n'est pas évident, une à une.

### 3a. `sAvailable` vs `sLeaseFrom`

In [ ]:
share_available_equals_start = (leases["sAvailable"] == leases["sLeaseFrom"]).mean()
print(f"sAvailable == sLeaseFrom dans {share_available_equals_start:.1%} des baux")

**Lecture :** dans la table des baux, `sAvailable` est une copie exacte de `sLeaseFrom`. Colonne redondante,
on l'ignore. (Dans `listings`, `sAvailable` a un autre sens : la date de disponibilité prochaine de l'unité.)

### 3b. `sTermSeq` vs `sRenewal`

Hypothèse : `sTermSeq` est le rang du bail **dans l'historique de l'unité** (1er, 2e, 3e bail...),
et non dans l'historique du locataire. Si c'est le cas, `sTermSeq = 1` est toujours une relocation,
et un `sTermSeq` élevé peut être un renouvellement ou un nouveau locataire.

In [ ]:
pd.crosstab(leases["sTermSeq"], leases["sRenewal"], margins=True)

**Lecture :** hypothèse confirmée. `sTermSeq = 1` est toujours une relocation (premier bail connu de l'unité),
et à partir de 2 on trouve les deux types. `sTermSeq` est donc le **rang du bail dans l'historique de l'unité**,
et c'est `sRenewal` qu'il faut utiliser pour séparer renouvellements et relocations.

### 3c. Durée du bail : `sTermMonths` et `sLeaseTerm`

Les durées non entières (5,9; 17,9; 23,9 mois) suggèrent une durée calculée à partir des dates
plutôt que saisie. Comme les baux n'ont pas tous 12 mois, toute comparaison de loyers doit être **annualisée**.

In [ ]:
print(leases["sTermMonths"].value_counts().to_string(), end="\n\n")
pd.crosstab(leases["sLeaseTerm"], leases["sTermMonths"])

**Lecture :** 95 % des baux durent 12 mois. Les étiquettes `sLeaseTerm` sont peu fiables
(« 6 Months » couvre aussi des baux de 12 mois, « Short Term » ne contient que des baux de 12 mois) :
on se fie à `sTermMonths`, jamais à `sLeaseTerm`.

### 3d. `sConcession` est-il un simple indicateur « loyer effectif < loyer contractuel »?

In [ ]:
effective_below_contract = leases["sRentEffective"] < leases["sRent"]
pd.crosstab(leases["sConcession"], effective_below_contract,
            rownames=["sConcession"], colnames=["sRentEffective < sRent"])

**Lecture :** correspondance parfaite. `sConcession` vaut 1 exactement quand le loyer effectif est inférieur
au loyer contractuel : c'est un indicateur dérivé, pas une information indépendante.

### 3e. `sRent` dans `listings` : de quel loyer s'agit-il?

`listings` contient aussi une colonne `sRent`, mais il n'y a qu'une ligne par unité : ce n'est pas un loyer de bail.
On la compare (1) au dernier loyer affiché de l'unité dans `asking` et (2) au loyer du dernier bail de l'unité.

In [ ]:
latest_asking = (asking.sort_values("month_start")
                 .groupby(UNIT_KEY).tail(1)[UNIT_KEY + ["sMonth", "sAskingRent"]])
latest_lease = (leases.sort_values("sLeaseFrom")
                .groupby(UNIT_KEY).tail(1)[UNIT_KEY + ["sRent"]]
                .rename(columns={"sRent": "last_lease_rent"}))

listing_comparison = (listings[UNIT_KEY + ["sRent", "sAsOf"]]
                      .merge(latest_asking, on=UNIT_KEY, how="left")
                      .merge(latest_lease, on=UNIT_KEY, how="left"))
equals_latest_asking = (listing_comparison["sRent"] - listing_comparison["sAskingRent"]).abs() < RENT_MATCH_TOLERANCE
equals_last_lease = (listing_comparison["sRent"] - listing_comparison["last_lease_rent"]).abs() < RENT_MATCH_TOLERANCE
print(f"sRent (listings) == dernier loyer affiché : {equals_latest_asking.mean():.1%}")
print(f"sRent (listings) == loyer du dernier bail : {equals_last_lease.mean():.1%}")

listing_markup_pct = (listing_comparison["sRent"] / listing_comparison["last_lease_rent"] - 1) * PCT
print("\nÉcart (%) entre sRent de listings et le loyer du dernier bail :")
print(listing_markup_pct.describe().round(2))

**Lecture :** `sRent` de `listings` n'est ni le dernier loyer affiché, ni le loyer du dernier bail.
Il est **systématiquement au-dessus** du loyer du dernier bail (entre 0 % et environ +5 %, médiane environ +2,5 %).
Hypothèse : c'est le **loyer proposé pour le prochain bail** de l'unité, en date de `sAsOf`.
Si elle se confirme, c'est un signal interne de la hausse visée par JADCO pour les prochains baux -
à rapprocher de la prévision 2026 (à utiliser avec prudence : un loyer proposé n'est pas un loyer signé).

## 4. La table des concessions

### 4a. Ce que contient chaque code

Pour chaque `sChargeCode` : combien, quel montant typique par période (`sAmount`), sur combien de mois (`sMonths`).

In [ ]:
concessions_by_code = (concessions.groupby("sChargeCode")
                        .agg(count=("sAmount", "size"),
                             median_amount=("sAmount", "median"),
                             median_months=("sMonths", "median"),
                             share_one_month=("sMonths", lambda months: (months == 1).mean()))
                        .sort_values("count", ascending=False))
concessions_by_code.round(2)

### 4b. `PromoPay` : rabais mensuel ou montant unique?

La description dit « mensuel ». Si c'était vrai, un montant proche du loyer signifierait que le résident
habite presque gratuitement. On compare `sAmount` au loyer du bail auquel la concession se rattache.

In [ ]:
def attach_concessions_to_leases(concession_df, lease_df):
    """Rattache chaque concession au bail de la même unité dont la période contient sDateFrom."""
    lease_columns = UNIT_KEY + ["sLeaseFrom", "sLeaseTo", "sRent", "sRentEffective",
                                "sTermMonths", "sRenewal", "lease_start_year"]
    candidates = (concession_df.reset_index(names="concession_id")
                  .merge(lease_df[lease_columns].reset_index(names="lease_id"), on=UNIT_KEY))
    starts_within_lease = candidates["sDateFrom"].between(candidates["sLeaseFrom"], candidates["sLeaseTo"])
    return candidates[starts_within_lease]


matched_concessions = attach_concessions_to_leases(concessions, leases)
print(f"concessions rattachées à un bail : {len(matched_concessions)} / {len(concessions)}")

promo = matched_concessions[matched_concessions["sChargeCode"] == "PromoPay"]
promo_months_of_rent = -promo["sAmount"] / promo["sRent"]
print(promo_months_of_rent.describe().round(2))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(promo_months_of_rent, bins=40)
ax.set_xlabel("montant PromoPay / loyer mensuel du bail (en mois de loyer)")
ax.set_ylabel("nombre de concessions")
ax.set_title("PromoPay vaut souvent près d'un mois de loyer complet")
plt.tight_layout()
plt.show()

**Lecture :** deux familles de concessions.

- **`PromoPay`** : inscrit sur un seul mois dans 88 % des cas, pour un montant médian d'environ 0,8 mois de loyer.
  Ce n'est **pas** un rabais mensuel, c'est un **rabais unique de signature** (l'équivalent d'un mois gratuit) -
  la description « mensuel » est fausse. Pour en mesurer l'effet sur le loyer, il faut l'étaler sur la durée du bail.
- **Tous les autres codes** : environ 130 à 150 $ par mois, typiquement sur 12 mois (parfois 3 ou 6) -
  ce sont de vrais rabais mensuels. `freepark`, `freelock` et `freeappl` portent sur des **accessoires**, pas sur le loyer.

### 4c. Les concessions non rattachées

Certaines concessions ne tombent dans aucun bail. On mesure de combien de jours elles précèdent
le début du bail suivant de la même unité.

In [ ]:
unmatched = concessions[~concessions.index.isin(matched_concessions["concession_id"])]

next_lease_start = (unmatched.reset_index(names="concession_id")
                    .merge(leases[UNIT_KEY + ["sLeaseFrom"]], on=UNIT_KEY)
                    .query("sLeaseFrom >= sDateFrom")
                    .groupby("concession_id")["sLeaseFrom"].min())
days_before_lease = (next_lease_start - unmatched["sDateFrom"].reindex(next_lease_start.index)).dt.days
print(f"concessions non rattachées : {len(unmatched)}")
print(days_before_lease.describe().round(1))

**Lecture :** les 204 concessions « orphelines » commencent toutes 1 à 30 jours **avant** le début d'un bail
de la même unité - vraisemblablement accordées entre la signature et l'emménagement. Pour un rattachement complet,
il faudrait élargir la fenêtre (par exemple à partir de `sSignDate`).

## 5. Peut-on reconstruire `sRentEffective` à partir des concessions?

C'est la vérification la plus importante : notre définition de la hausse repose sur `sRentEffective`.

Hypothèse : loyer effectif = loyer contractuel + (somme des concessions du bail, `sAmount × sMonths`) / durée du bail.
Les montants de concessions sont négatifs, donc le loyer effectif baisse.

In [ ]:
matched_concessions = matched_concessions.assign(
    total_amount=matched_concessions["sAmount"] * matched_concessions["sMonths"],
    is_accessory=matched_concessions["sChargeCode"].isin(ACCESSORY_CHARGE_CODES))

concession_totals = matched_concessions.groupby("lease_id").agg(
    concession_total=("total_amount", "sum"),
    concession_count=("total_amount", "size"))

reconstruction = leases.join(concession_totals).fillna({"concession_total": 0, "concession_count": 0})
reconstruction["observed_gap"] = reconstruction["sRentEffective"] - reconstruction["sRent"]
reconstruction["predicted_gap"] = reconstruction["concession_total"] / reconstruction["sTermMonths"]
reconstruction["reconstruction_error"] = reconstruction["observed_gap"] - reconstruction["predicted_gap"]
reconstruction["is_reconstructed"] = reconstruction["reconstruction_error"].abs() < RENT_MATCH_TOLERANCE

print(f"baux dont sRentEffective est reconstruit exactement : {reconstruction['is_reconstructed'].mean():.1%}")
print()
print("Cas d'échec :")
has_gap = reconstruction["observed_gap"] < 0
has_concession_rows = reconstruction["concession_count"] > 0
failure_cases = pd.crosstab(has_gap[~reconstruction["is_reconstructed"]],
                            has_concession_rows[~reconstruction["is_reconstructed"]],
                            rownames=["rabais observé"], colnames=["concessions trouvées"])
print(failure_cases)

In [ ]:
reconstruction_rate_by_year = reconstruction.groupby("lease_start_year")["is_reconstructed"].mean()
(reconstruction_rate_by_year * PCT).round(1).rename("pct_reconstructed").to_frame().T

**Lecture :** la formule reconstruit exactement `sRentEffective` pour environ 62 % des baux - donc `sRentEffective`
**est bien** le loyer contractuel diminué des concessions étalées sur la durée du bail, y compris `PromoPay` étalé.
Les échecs ont trois causes :

1. **497 baux avec un rabais, mais aucune concession dans la table** : la table des concessions est incomplète
   (c'est un extrait).
2. **264 baux avec des concessions, mais aucun rabais** : surtout des accessoires (`freepark`, `Freeothe`, `freelock`).
   Ces concessions ne sont donc pas toujours déduites du loyer effectif.
3. **871 baux où les deux existent mais ne concordent pas** : rattachement imparfait (voir 4c) ou règles de calcul
   que l'extrait ne permet pas de retrouver.

**Décision :** on utilise `sRentEffective` tel quel, comme chiffre de référence calculé par Yardi, plutôt que de le
recalculer à partir d'une table incomplète. La table des concessions sert seulement à expliquer la **composition**
des rabais (quels types, quels montants).

## 6. Effet de composition (mix)

Pourquoi la médiane annuelle de tous les baux ne mesure pas la hausse : on regarde ce qui change
dans *qui* est loué chaque année.

In [ ]:
analysis_window = leases[leases["lease_start_year"].between(FIRST_ANALYSIS_YEAR, LAST_OBSERVED_YEAR)].copy()
analysis_window["rent_per_sqft"] = analysis_window["sRent"] / analysis_window["sSqft"]

TWO_BEDROOMS = 2
mix_by_year = analysis_window.groupby("lease_start_year").agg(
    leases=("sRent", "size"),
    buildings=("sBuilding", "nunique"),
    median_rent=("sRent", "median"),
    median_sqft=("sSqft", "median"),
    pct_two_bed_plus=("sBeds", lambda beds: (beds >= TWO_BEDROOMS).mean() * PCT),
    median_rent_per_sqft=("rent_per_sqft", "median"))
mix_by_year.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

share_by_building = pd.crosstab(analysis_window["lease_start_year"], analysis_window["sBuilding"], normalize="index") * PCT
share_by_building.plot(kind="bar", stacked=True, ax=axes[0], width=0.85)
axes[0].set_ylabel("% des baux de l'année")
axes[0].set_xlabel("année de début du bail")
axes[0].set_title("Le portefeuille change de composition en 2023-2024")
axes[0].legend(fontsize=8, loc="upper left", bbox_to_anchor=(1, 1))

rent_per_sqft_by_building = analysis_window.groupby("sBuilding")["rent_per_sqft"].median().sort_values()
rent_per_sqft_by_building.plot(kind="barh", ax=axes[1])
axes[1].set_xlabel("loyer médian au pi² ($)")
axes[1].set_title("Les nouveaux immeubles sont plus chers au pi²")
plt.tight_layout()
plt.show()

**Lecture :** de 3 immeubles jusqu'en 2022, on passe à 5 en 2023 puis 6 en 2024. Les unités louées deviennent
plus petites (superficie médiane de 1 175 à 1 025 pi²) et moins souvent des 2 chambres et plus (de 78 % à 60 %),
mais beaucoup plus chères au pi². La médiane annuelle monte donc parce que **le portefeuille change**, pas
parce qu'un même appartement coûte plus cher : c'est l'effet de mix.

## 7. Les concessions dans le temps

Le rabais est mesuré comme `1 - sRentEffective / sRent` : la part du loyer contractuel qui n'est pas perçue.

In [ ]:
analysis_window["discount_pct"] = (1 - analysis_window["sRentEffective"] / analysis_window["sRent"]) * PCT
analysis_window["lease_type"] = analysis_window["sRenewal"].map({RENEWAL: "renewal", TURNOVER: "turnover"})

concessions_by_year = analysis_window.pivot_table(
    index="lease_start_year", columns="lease_type",
    values=["sConcession", "discount_pct"], aggfunc="mean")
concessions_by_year["sConcession"] *= PCT
concessions_by_year.round(2)

In [ ]:
discount_by_building_year = analysis_window.pivot_table(
    index="lease_start_year", columns="sBuilding", values="discount_pct", aggfunc="mean")

fig, ax = plt.subplots(figsize=(10, 4.5))
discount_by_building_year.plot(ax=ax, marker="o")
ax.set_ylabel("rabais moyen (% du loyer contractuel)")
ax.set_xlabel("année de début du bail")
ax.set_title("Les concessions augmentent partout, pas seulement dans les nouveaux immeubles")
ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

**Lecture :** le rabais moyen passe d'environ 2 % en 2019-2020 à environ 9 % en 2025, et la part des baux avec
concession d'environ 50 % à plus de 80 %. Les **renouvellements** en reçoivent autant que les relocations.
C'est ce qui fait diverger croissance contractuelle et croissance effective à partir de 2024 : les hausses
inscrites au bail sont en partie redonnées sous forme de concessions.

## 8. Test : les unités restées vides plus longtemps reçoivent-elles plus de concessions?

Intuition : un propriétaire qui peine à relouer offre des mois gratuits.

**Attention :** ce test se fait **unité par unité** (jours entre la fin d'un bail et le début du suivant).
On n'en tire **aucun taux d'inoccupation** : l'extrait ne contient pas toutes les unités, un tel taux
serait un artefact. Pour l'inoccupation du marché, on utilisera la SCHL (notebook final, données externes).

In [ ]:
leases_sorted = leases.sort_values(UNIT_KEY + ["sLeaseFrom"]).copy()
leases_sorted["previous_lease_end"] = leases_sorted.groupby(UNIT_KEY)["sLeaseTo"].shift(1)
leases_sorted["days_between_leases"] = (leases_sorted["sLeaseFrom"] - leases_sorted["previous_lease_end"]).dt.days
leases_sorted["discount_pct"] = (1 - leases_sorted["sRentEffective"] / leases_sorted["sRent"]) * PCT

with_previous = leases_sorted.dropna(subset=["days_between_leases"]).copy()
downtime_bins = DOWNTIME_BINS_DAYS
with_previous["downtime_bucket"] = pd.cut(with_previous["days_between_leases"], bins=downtime_bins)

print("Jours entre deux baux, par type de bail :")
print(with_previous.groupby("sRenewal")["days_between_leases"].describe().round(1), end="\n\n")

turnovers = with_previous[with_previous["sRenewal"] == TURNOVER]
turnovers.groupby("downtime_bucket", observed=True).agg(
    leases=("sRent", "size"),
    pct_with_concession=("sConcession", lambda flag: flag.mean() * PCT),
    mean_discount_pct=("discount_pct", "mean")).round(2)

**Lecture :** résultat négatif, et c'est une information utile.

- Le délai entre deux baux a **exactement la même distribution** pour les renouvellements et les relocations
  (médiane de 1 jour). Or un renouvellement n'a pas de période vide : ce délai ne mesure donc pas une vraie vacance.
- La part de baux avec concession ne varie pas avec ce délai (environ 61-63 %), sauf dans la dernière tranche,
  trop petite (34 baux) pour conclure.
- Combiné à la section 7 (les renouvellements reçoivent autant de concessions), cela indique que les concessions
  sont une **politique commerciale globale**, liée au contexte de marché, et non une réaction à une unité restée vide.

**Suite :** tester la relation avec le **taux d'inoccupation du marché publié par la SCHL** (RMR de Montréal et d'Ottawa)
dans le notebook final.

## 9. Historique des loyers affichés

Combien d'observations par unité, et sont-elles mensuelles en continu?

In [ ]:
observations_per_unit = asking.groupby(UNIT_KEY).size()
print("observations par unité :")
print(observations_per_unit.describe().round(1), end="\n\n")

asking_sorted = asking.sort_values(UNIT_KEY + ["month_start"])
months_between_observations = (asking_sorted.groupby(UNIT_KEY)["month_start"].diff().dt.days / 30.44).round()
print("mois entre deux observations consécutives d'une même unité :")
print(months_between_observations.value_counts().sort_index().head(15).to_string())

**Lecture :** les loyers affichés ne forment pas une série mensuelle continue. Les observations arrivent soit
en mois consécutifs (une unité affichée pendant sa mise en marché), soit à environ 12 mois d'écart (une
observation par cycle de location). C'est donc un **loyer demandé au moment de la remise en marché**, pas un suivi
mensuel du marché, et on ne peut pas en déduire une durée d'inoccupation.

## 10. Qualité des données

Loyers nuls ou extrêmes, et taille des échantillons par immeuble et par année
(un chiffre calculé sur trop peu de baux n'est pas fiable).

In [ ]:
print("loyers contractuels <= 0 :", (leases["sRent"] <= 0).sum())
print("loyers effectifs <= 0 :", (leases["sRentEffective"] <= 0).sum())
print("loyers effectifs > contractuels :", (leases["sRentEffective"] > leases["sRent"]).sum())
print()
print(leases[["sRent", "sRentEffective", "sSqft"]].describe().round(1))

In [ ]:
# Nombre de baux par immeuble, année et type : les cellules à faible effectif
# donneront des médianes instables si on prévoit par immeuble.
pd.crosstab(analysis_window["sBuilding"],
            [analysis_window["lease_start_year"], analysis_window["lease_type"]])

**Lecture :** aucun loyer nul ou négatif, et aucun loyer effectif supérieur au contractuel. Les effectifs sont faibles
pour les renouvellements des nouveaux immeubles la première année (Le Carlyle en 2023, Westpark en 2024) : c'est normal,
leurs premiers locataires n'avaient pas encore fini leur bail. Une prévision par immeuble devra en tenir compte
(regrouper, ou s'appuyer davantage sur la tendance du portefeuille quand l'échantillon est petit).

## 11. Conclusions et renommage proposé

| Constat | Conséquence pour la suite |
|---|---|
| `sPropCode + sUnitCode` = `hUnit` = `sLink` (1 pour 1); `sSite + sUnitCode` provoque 248 collisions | Appariement sur `UNIT_KEY` |
| `sAvailable` (baux) = copie de `sLeaseFrom` | Colonne ignorée |
| `sTermSeq` = rang du bail dans l'historique de l'unité | Séparer les types de baux avec `sRenewal`, pas `sTermSeq` |
| Durées de 6 à 24 mois; `sLeaseTerm` incohérent | Annualiser les variations; ignorer `sLeaseTerm` |
| `sConcession` = indicateur `sRentEffective < sRent` | Pas d'information propre |
| `sRent` de `listings` = environ +2,5 % au-dessus du dernier bail | Probable loyer proposé au prochain bail : signal interne pour 2026 |
| `PromoPay` = rabais **unique** d'environ un mois de loyer, pas mensuel | Renommé; étalé sur la durée du bail |
| `sRentEffective` = contractuel - concessions étalées (vérifié sur 62 % des baux, table des concessions incomplète) | `sRentEffective` utilisé tel quel |
| Médiane annuelle tirée par l'arrivée de 3 immeubles haut de gamme | Croissance à unité constante |
| Rabais moyen d'environ 2 % (2020) à environ 9 % (2025), renouvellements inclus | Explique la divergence contractuel / effectif |
| Aucun lien interne vacance -> concession | Utiliser l'inoccupation de la SCHL comme contexte |
| `asking` = loyer demandé à la remise en marché, pas une série mensuelle | Comparer au loyer signé des relocations |

### Renommage proposé

Noms qui décrivent ce que les colonnes contiennent réellement. Ce dictionnaire sera repris dans le notebook final.

In [ ]:
LEASE_COLUMN_RENAMES = {
    "sRent": "contract_rent",
    "sRentEffective": "effective_rent",            # contractuel - concessions étalées sur le bail
    "sConcession": "has_concession",               # = effective_rent < contract_rent
    "sRenewal": "is_renewal",                      # 1 = locataire en place, 0 = relocation
    "sTermSeq": "unit_lease_rank",                 # rang du bail dans l'historique de l'unité
    "sTermMonths": "lease_term_months",
    "sLeaseFrom": "lease_start",
    "sLeaseTo": "lease_end",
    "sSignDate": "sign_date",
    "sAvailable": "lease_start_duplicate",         # copie exacte de sLeaseFrom
    "sLeaseTerm": "lease_term_label_unreliable",
}
LISTING_COLUMN_RENAMES = {
    "sRent": "proposed_next_rent",                 # hypothèse : loyer proposé au prochain bail
    "sAsOf": "proposed_rent_as_of",
    "sAvailable": "next_available_date",
}
ASKING_COLUMN_RENAMES = {"sAskingRent": "asking_rent_at_remarketing"}
CHARGE_CODE_LABELS = {
    "PromoPay": "one_time_signing_discount",       # PAS mensuel : environ un mois de loyer, une fois
    "freerent": "free_rent_monthly",
    "freepark": "free_parking_accessory",
    "freelock": "free_locker_accessory",
    "freeappl": "free_appliances_accessory",
    "Freeothe": "other_uncategorized",
    "Indem": "tenant_indemnity",
    "Referenc": "referral_credit",
}

leases_renamed = leases.rename(columns=LEASE_COLUMN_RENAMES)
leases_renamed[list(LEASE_COLUMN_RENAMES.values())].head()